# NACC Horizon Sweep (Counts Only, Original Cohort Logic)

This notebook mirrors the **visit date** + **baseline selection** logic from your original cohort notebook, but it only produces **counts/tables** to help you choose:
- a better follow-up horizon (3y, 4y, 5y, 6y, …)
- and the horizon that maximizes **PD/LBD** and **Vascular** dementia event counts.

✅ **No cohort.csv is written.**


## 0) Setup
Edit paths/flags here.


In [53]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# --- PATHS: update to your local paths if needed ---
# If you keep the same layout as your original notebook:
DATA_DIR = os.path.join('..', 'data')
OUT_DIR  = os.path.join('..', 'output')
os.makedirs(OUT_DIR, exist_ok=True)

# Main visit-level file (the one with repeated rows per NACCID)
NACC_MAIN_PATH = os.path.join(DATA_DIR, 'investigator_nacc72.csv')

# --- Options ---
USE_ONLY_V3 = False   # True => keep only FORMVER in {3, 3.2}

# Horizon sweep range
MIN_H = 3
MAX_H = 12   # increase if you want; 12 is usually plenty


## 1) Load data + construct visit_date (same as your original notebook)
Your original notebook builds `visit_date` from `VISITYR`, `VISITMO`, `VISITDAY` with sensible defaults.


In [54]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df['NACCID'] = df['NACCID'].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_['VISITYR'], errors='coerce')
    m = pd.to_numeric(df_['VISITMO'], errors='coerce').fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_['VISITDAY'], errors='coerce').fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors='coerce')

df['visit_date'] = make_visit_date(df)
df = df.dropna(subset=['NACCID', 'visit_date']).copy()
df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)

if 'FORMVER' in df.columns:
    df['FORMVER'] = pd.to_numeric(df['FORMVER'], errors='coerce')

print('Rows:', len(df), 'Participants:', df['NACCID'].nunique())
df[['NACCID','visit_date'] + ([ 'FORMVER' ] if 'FORMVER' in df.columns else [])].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_38920\1939311393.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['visit_date'] = make_visit_date(df)


Rows: 207454 Participants: 55268


,NACCID,visit_date,FORMVER
0,NACC000011,2006-04-17,1.0
1,NACC000011,2007-06-18,1.0
2,NACC000011,2008-06-03,2.0
3,NACC000011,2009-08-03,2.0
4,NACC000034,2015-07-16,3.0


## 2) Optional FORMVER filter (same as original)


In [55]:
USE_ONLY_V3 = False  # set True to keep only FORMVER in {3, 3.2}

if USE_ONLY_V3 and 'FORMVER' in df.columns:
    before = len(df)
    df = df[df['FORMVER'].isin([3, 3.2])].copy()
    df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)
    print('Filtered to FORMVER in {3,3.2}: rows', before, '->', len(df))
else:
    print('No FORMVER filtering applied.')

No FORMVER filtering applied.


## 3) Baseline selection (same logic as your original notebook)
Baseline is the **earliest visit** where:
- cognitive status is **non-demented** (CN/MCI-ish), AND
- **DEMENTED==0** (if available)


In [56]:
import numpy as np
import pandas as pd

def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

# DEMENTED: 0/1
no_dem_mask = (df["DEMENTED"] == 0) if "DEMENTED" in df.columns else pd.Series(True, index=df.index)

# Treat these as "not usable" / missing-ish if present
MISSING_CODES = {-4, 9}

if "NACCUDSD" in df.columns:
    # Primary baseline definition: CN/MCI (not dementia)
    uds_non_dem = df["NACCUDSD"].isin([1, 2, 3])

    # Fallback for rows where NACCUDSD is missing or coded as missing-ish
    if "COGSTAT" in df.columns:
        uds_bad = df["NACCUDSD"].isna() | df["NACCUDSD"].isin(list(MISSING_CODES))
        cog_non_dem = df["COGSTAT"].isin([1, 2, 3, 4])  # clinician judged non-demented range
        cnmci_mask = uds_non_dem | (uds_bad & cog_non_dem)
    else:
        cnmci_mask = uds_non_dem

else:
    # If NACCUDSD not present, use NORMCOG / NACCTMCI
    cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    cnmci_mask = cn_mask | mci_mask

    # Optional fallback if even those are missing
    if "COGSTAT" in df.columns:
        cog_non_dem = df["COGSTAT"].isin([1,2,3,4])
        cnmci_mask = cnmci_mask | cog_non_dem

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID","baseline_date"]].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_38920\72823028.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


Baseline participants: 38341


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


In [57]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

#DEMENTED: 0-No, 1-Yes
#NACCUDSD:1-Normal, 2-Impaired not MCI, 3-MCI, 4-Dementia
#COGSTAT: 0-Unable to render, 1-Better than normal age, 2-Normal for age, 3-one or more test cases abnormal
#4-three or more test cases abnormal,9-missing, -4 not available
#NORMCOG:1- Normal, 0-significant impairment
# NACCTMCI
# 1 = Amnestic MCI- single domain
# 2 = Amnestic MCI- multiple domain
# 3 = Non-amnestic MCI- single domain
# 4 = Non-amnestic MCI- multiple domain
# 8 = No diagnosis of MCI
# Dementia flag (best available)
if "DEMENTED" in df.columns:
    no_dem_mask = (df["DEMENTED"] == 0)
else:
    no_dem_mask = pd.Series(True, index=df.index)

# --- Baseline eligibility: NOT demented, and CN/MCI-ish if we can tell ---
if "NACCUDSD" in df.columns:
    # Best: UDS-derived cognitive status
    cnmci_mask = df["NACCUDSD"].isin([1, 2, 3])
else:
    # Fallback: use NORMCOG and/or NACCTMCI if present
    cn_mask = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1, 2, 3, 4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    # If neither exists, allow any non-demented as baseline (still consistent with goal)
    if ("NORMCOG" in df.columns) or ("NACCTMCI" in df.columns):
        cnmci_mask = cn_mask | mci_mask
    else:
        cnmci_mask = pd.Series(True, index=df.index)

# Optional: remove obviously unusable COGSTAT entries (but DO NOT define CN/MCI using it)
if "COGSTAT" in df.columns:
    cog_ok = ~df["COGSTAT"].isin([0, 9, -4])
    # You can choose to apply or not; I usually DO NOT apply for baseline selection
    # cnmci_mask = cnmci_mask & cog_ok

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = (baseline_candidates
            .sort_values(["NACCID", "visit_date"])
            .groupby("NACCID", as_index=False)
            .first())

baseline["baseline_date"] = baseline["visit_date"]
baseline = baseline.drop(columns=["visit_date"])

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID", "baseline_date"]].head()

C:\Users\branjan\AppData\Local\Temp\ipykernel_38920\2496616421.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first())


Baseline participants: 38341


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


## 4) Follow-up time & first dementia event (no horizon restriction yet)
We compute time since baseline for every visit, then identify each participant’s **first post-baseline dementia visit**.


In [58]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline['NACCID'])
df2 = df[df['NACCID'].isin(keep_ids)].merge(
    baseline[['NACCID','baseline_date']], on='NACCID', how='left'
).copy()

df2['t_years'] = years_between(df2['baseline_date'], df2['visit_date'])
post = df2[df2['t_years'] > 0].copy()

print("=== CHECKPOINT: post ===")
print("baseline N:", baseline["NACCID"].nunique())
print("post rows:", len(post), " post unique IDs:", post["NACCID"].nunique())
print("post visit_date min/max:", post["visit_date"].min(), post["visit_date"].max())
print("t_years min/max:", post["t_years"].min(), post["t_years"].max())

max_follow = post.groupby('NACCID')['t_years'].max().rename('max_follow_years')
baseline2 = baseline.merge(max_follow, on='NACCID', how='left')

print("\n=== CHECKPOINT: follow-up ===")
print("max_follow_years describe:")
display(baseline2["max_follow_years"].describe())
print("eligible @4y:", (baseline2["max_follow_years"] >= 4).sum())
print("eligible @5y:", (baseline2["max_follow_years"] >= 5).sum())

baseline2['max_follow_years'] = baseline2['max_follow_years'].fillna(0)

#DEMENTED: 0-No, 1-Yes
#NACCUDSD:1-Normal, 2-Impaired not MCI, 3-MCI, 4-Dementia
if 'NACCUDSD' in post.columns:
    dem_events_all = post[post['NACCUDSD'] == 4].copy()
    dementia_def = 'NACCUDSD==4'
else:
    dem_events_all = post[post['DEMENTED'] == 1].copy()
    dementia_def = 'DEMENTED==1'

first_dem_ever = (dem_events_all.sort_values(['NACCID','visit_date'])
                           .groupby('NACCID', as_index=False)
                           .first()
                           .rename(columns={'visit_date':'dementia_date'}))

first_dem_ever = first_dem_ever.merge(
    baseline2[['NACCID', 'baseline_date', 'max_follow_years']],
    on='NACCID',
    how='left',
    suffixes=('', '_base')   # prevent baseline_date_x/y
)

# If baseline_date got duplicated somehow, normalize it:
if 'baseline_date' not in first_dem_ever.columns:
    for c in ['baseline_date_base', 'baseline_date_x', 'baseline_date_y']:
        if c in first_dem_ever.columns:
            first_dem_ever['baseline_date'] = first_dem_ever[c]
            break
        
first_dem_ever['time_to_dem_years'] = years_between(first_dem_ever['baseline_date'], first_dem_ever['dementia_date'])

print("\n=== CHECKPOINT: first_dem_ever ===")
print("first_dem_ever unique IDs:", first_dem_ever["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_ever["time_to_dem_years"].min(),
      first_dem_ever["time_to_dem_years"].median(),
      first_dem_ever["time_to_dem_years"].max())

print('Dementia event definition:', dementia_def)
print('Participants with ANY post-baseline dementia event:', first_dem_ever['NACCID'].nunique())
first_dem_ever[['NACCID','baseline_date','dementia_date','time_to_dem_years']].head()

=== CHECKPOINT: post ===
baseline N: 38341
post rows: 123654  post unique IDs: 27703
post visit_date min/max: 2006-04-03 00:00:00 2025-10-21 00:00:00
t_years min/max: 0.008213552361396304 19.392197125256672

=== CHECKPOINT: follow-up ===
max_follow_years describe:


count    27703.000000
mean         5.499126
std          4.268776
min          0.109514
25%          2.080767
50%          4.301164
75%          7.685147
max         19.392197
Name: max_follow_years, dtype: float64

eligible @4y: 14853
eligible @5y: 12474

=== CHECKPOINT: first_dem_ever ===
first_dem_ever unique IDs: 5502
time_to_dem_years min/median/max: 0.4188911704312115 2.6885694729637235 19.304585900068446
Dementia event definition: NACCUDSD==4
Participants with ANY post-baseline dementia event: 5502


C:\Users\branjan\AppData\Local\Temp\ipykernel_38920\3760630362.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


,NACCID,baseline_date,dementia_date,time_to_dem_years
0,NACC000034,2015-07-16,2021-05-21,5.848049
1,NACC000162,2009-03-10,2017-11-10,8.670773
2,NACC000225,2009-09-23,2017-07-07,7.786448
3,NACC000271,2007-06-20,2009-05-14,1.900068
4,NACC000333,2021-05-13,2024-07-24,3.197810


In [59]:
HORIZON_YEARS = 5  # or 4

window = post[post["t_years"] <= HORIZON_YEARS].copy()

if "NACCUDSD" in window.columns:
    dem_events_H = window[window["NACCUDSD"] == 4].copy()
else:
    dem_events_H = window[window["DEMENTED"] == 1].copy()

first_dem_within_H = (dem_events_H.sort_values(["NACCID", "visit_date"])
                          .groupby("NACCID", as_index=False)
                          .first()
                          .rename(columns={"visit_date": "dementia_date"}))

# Merge ONLY max follow-up (avoid baseline_date duplication)
first_dem_within_H = first_dem_within_H.merge(
    baseline2[["NACCID", "max_follow_years"]],
    on="NACCID", how="left"
)

# Compute time-to-dementia using the baseline_date already in first_dem_within_H
# (it may be called baseline_date or baseline_date_x depending on your current state)
if "baseline_date" not in first_dem_within_H.columns:
    # prefer baseline_date_x if it exists
    if "baseline_date_x" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_x": "baseline_date"})
    elif "baseline_date_y" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_y": "baseline_date"})
    else:
        raise KeyError("No baseline_date column found in first_dem_within_H.")

first_dem_within_H["time_to_dem_years"] = years_between(first_dem_within_H["baseline_date"], first_dem_within_H["dementia_date"])

print("Participants with dementia within", HORIZON_YEARS, "y:", first_dem_within_H["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_within_H["time_to_dem_years"].min(),
      first_dem_within_H["time_to_dem_years"].median(),
      first_dem_within_H["time_to_dem_years"].max())

Participants with dementia within 5 y: 3958
time_to_dem_years min/median/max: 0.4188911704312115 2.005475701574264 4.999315537303217


C:\Users\branjan\AppData\Local\Temp\ipykernel_38920\1069559368.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


## 5) Etiology subtype mapping (event-level)
This uses **primary etiology if available** (recommended for a clean 3-class label):
- PD_LBD primary: `NACCLBDP == 1`
- Vascular primary: `CVDIF == 1`
- AD primary: `NACCALZP == 1` (or fallback `NACCALZD == 1`)

We also compute an **"any mention"** version (etiology present, regardless of primary/contributing) to see how much you gain.


# Section 1A:
Changes in building cohort logic to increase the sample counts in the 4 classifiers:
Updated the logic of classifying and assigning the output etiology labels (AD, PD_LBD, Vascular, No_Dementia) using 3 level priority
Primary (strict)
Contributing (only if still Other/Unk after primary), then Contributing flags are checked to classify the final etiology.
Any-mention (only if still Other/Unk after contributing), then other secondary flags are checked to classify the final etiology.


In [60]:

import numpy as np
import pandas as pd


def subtype_primary(row):

    formver = row.get("FORMVER", np.nan)

    # ------------------
    # PD / LBD
    # ------------------
    if pd.notna(row.get("NACCLBDP")) and int(row["NACCLBDP"]) == 1:
        return "PD_LBD"

    # ------------------
    # Vascular
    # ------------------
    if formver >= 3:
        if pd.notna(row.get("CVDIF")) and int(row["CVDIF"]) == 1:
            return "Vascular"
    else:
        if pd.notna(row.get("VASCIF")) and int(row["VASCIF"]) == 1:
            return "Vascular"
        if pd.notna(row.get("VASCPSIF")) and int(row["VASCPSIF"]) == 1:
            return "Vascular"

    # ------------------
    # AD
    # ------------------
    if pd.notna(row.get("NACCALZP")) and int(row["NACCALZP"]) == 1:
        return "AD"
    if pd.notna(row.get("NACCALZD")) and int(row["NACCALZD"]) == 1:
        return "AD"

    return "Other/Unk"



def _as_int(x):
    # safe int conversion for strings/float/NA
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        return None

def subtype_anymention(row) -> str:
    formver = row.get("FORMVER", np.nan)
    formver = float(formver) if pd.notna(formver) else np.nan

    # ------------------
    # PD / LBD (any mention)
    # ------------------
    # Etiology present (preferred)
    v = _as_int(row.get("NACCLBDE"))
    if v == 1:
        return "PD_LBD"

    # Optional extra-inclusive: syndrome present (can increase counts)
    # If you want the loosest any-mention, uncomment:
    # v = _as_int(row.get("NACCLBDS"))
    # if v == 1:
    #     return "PD_LBD"

    # ------------------
    # Vascular (any mention)
    # ------------------
    if pd.notna(formver) and formver >= 3:
        # UDS v3+: vascular brain injury etiology present
        v = _as_int(row.get("CVD"))
        if v == 1:
            return "Vascular"
    else:
        # UDS v1.2 / v2: vascular dementia etiology flags (probable/possible)
        v = _as_int(row.get("VASC"))
        if v == 1:
            return "Vascular"
        v = _as_int(row.get("VASCPS"))
        if v == 1:
            return "Vascular"

    # ------------------
    # AD (any mention)
    # ------------------
    v = _as_int(row.get("NACCALZD"))
    if v == 1:
        return "AD"

    return "Other/Unk"

# === Subtype labeling at dementia event (v1/v2/v3 compatible) ===
# Uses functions subtype_primary() and subtype_anymention() defined above.

def subtype_contributing(row) -> str:
    """Contributing etiology ONLY (used as fallback when primary is Other/Unk)."""
    formver = row.get("FORMVER", np.nan)
    try:
        formver = float(formver) if pd.notna(formver) else np.nan
    except Exception:
        formver = np.nan

    # PD/LBD contributing
    if _as_int(row.get("NACCLBDP")) == 2:
        return "PD_LBD"

    # Vascular contributing
    if pd.notna(formver) and formver >= 3:
        if _as_int(row.get("CVDIF")) == 2:
            return "Vascular"
    else:
        if _as_int(row.get("VASCIF")) == 2:
            return "Vascular"
        if _as_int(row.get("VASCPSIF")) == 2:
            return "Vascular"

    # AD contributing (etiology primary/contrib field exists)
    if _as_int(row.get("NACCALZP")) == 2:
        return "AD"

    return "Other/Unk"


# Stage 1: Primary
first_dem_ever["subtype_primary"] = first_dem_ever.apply(subtype_primary, axis=1)

# Stage 2: Contributing (computed for all, but used only if primary is Other/Unk)
first_dem_ever["subtype_contrib"] = first_dem_ever.apply(subtype_contributing, axis=1)

# Stage 3: Any-mention
first_dem_ever["subtype_any"] = first_dem_ever.apply(subtype_anymention, axis=1)

# Final label: Primary -> Contributing -> Any-mention
first_dem_ever["subtype_primary_contrib_any"] = np.where(
    first_dem_ever["subtype_primary"].ne("Other/Unk"),
    first_dem_ever["subtype_primary"],
    np.where(
        first_dem_ever["subtype_contrib"].ne("Other/Unk"),
        first_dem_ever["subtype_contrib"],
        first_dem_ever["subtype_any"]
    )
)

# Label source (nice for reporting)
first_dem_ever["subtype_label_source_3tier"] = np.select(
    [
        first_dem_ever["subtype_primary"].ne("Other/Unk"),
        first_dem_ever["subtype_primary"].eq("Other/Unk") & first_dem_ever["subtype_contrib"].ne("Other/Unk"),
        first_dem_ever["subtype_primary"].eq("Other/Unk") & first_dem_ever["subtype_contrib"].eq("Other/Unk") & first_dem_ever["subtype_any"].ne("Other/Unk"),
    ],
    ["primary", "fallback_contributing", "fallback_anymention"],
    default="other/unk"
)

print("3-tier label source counts:")
display(first_dem_ever["subtype_label_source_3tier"].value_counts())

print("\nFinal (Primary -> Contrib -> Any) subtype counts:")
display(first_dem_ever["subtype_primary_contrib_any"].value_counts())

rescued_by_contrib = (
    first_dem_ever["subtype_primary"].eq("Other/Unk")
    & first_dem_ever["subtype_contrib"].ne("Other/Unk")
).sum()

rescued_by_any = (
    first_dem_ever["subtype_primary"].eq("Other/Unk")
    & first_dem_ever["subtype_contrib"].eq("Other/Unk")
    & first_dem_ever["subtype_any"].ne("Other/Unk")
).sum()

print("Rescued by contributing:", rescued_by_contrib)
print("Rescued by any-mention (after contrib):", rescued_by_any)


3-tier label source counts:


subtype_label_source_3tier
primary                  4932
other/unk                 528
fallback_contributing      37
fallback_anymention         5
Name: count, dtype: int64


Final (Primary -> Contrib -> Any) subtype counts:


subtype_primary_contrib_any
AD           4304
Other/Unk     528
PD_LBD        384
Vascular      286
Name: count, dtype: int64

Rescued by contributing: 37
Rescued by any-mention (after contrib): 5


## 6) Horizon sweep tables
For each horizon H:
- how many baseline participants have **≥ H years follow-up**
- how many have **dementia by H years**
- dementia-by-H counts by subtype (primary and any-mention)


In [61]:
# cols = ["NACCALZP","NACCLBDP","CVDIF","VASCIF","VASCPSIF","NACCALZD","NACCLBDE","CVD","VASC","VASCPS"]
# present_cols = [c for c in cols if c in first_dem_ever.columns]

# first_dem_ever.loc[first_dem_ever["subtype_primary_contrib_any"].eq("Other/Unk"), present_cols].isna().mean().sort_values()

In [62]:
# for c in ["NACCALZP","NACCLBDP","CVDIF","VASCIF","VASCPSIF"]:
#     if c in first_dem_ever.columns:
#         print(c, (first_dem_ever.loc[first_dem_ever["subtype_primary_contrib_any"].eq("Other/Unk"), c] == 7).sum())

In [63]:
# Apply labels on the 'first dementia ever' table
first_dem_ever["subtype_primary"] = first_dem_ever.apply(subtype_primary, axis=1)
first_dem_ever["subtype_any"]     = first_dem_ever.apply(subtype_anymention, axis=1)
first_dem_ever["subtype_contrib"] = first_dem_ever.apply(subtype_contributing, axis=1)

# 3-tier: Primary -> Contributing -> Any-mention
first_dem_ever["subtype_primary_contrib"] = np.where(
    first_dem_ever["subtype_primary"].eq("Other/Unk"),
    first_dem_ever["subtype_contrib"],
    first_dem_ever["subtype_primary"],
)

first_dem_ever["subtype_primary_contrib_any"] = np.where(
    first_dem_ever["subtype_primary_contrib"].eq("Other/Unk"),
    first_dem_ever["subtype_any"],
    first_dem_ever["subtype_primary_contrib"],
)

# Track where the final label came from
first_dem_ever["subtype_label_source_3tier"] = "primary"
first_dem_ever.loc[
    first_dem_ever["subtype_primary"].eq("Other/Unk") & ~first_dem_ever["subtype_contrib"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "fallback_contributing"
first_dem_ever.loc[
    first_dem_ever["subtype_primary_contrib"].eq("Other/Unk") & ~first_dem_ever["subtype_any"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "fallback_anymention"
first_dem_ever.loc[
    first_dem_ever["subtype_primary_contrib_any"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "other/unk"

print("3-tier label source counts:")
display(first_dem_ever["subtype_label_source_3tier"].value_counts())

print("\nFinal (Primary -> Contrib -> Any) subtype counts:")
display(first_dem_ever["subtype_primary_contrib_any"].value_counts())


3-tier label source counts:


subtype_label_source_3tier
primary                  4932
other/unk                 528
fallback_contributing      37
fallback_anymention         5
Name: count, dtype: int64


Final (Primary -> Contrib -> Any) subtype counts:


subtype_primary_contrib_any
AD           4304
Other/Unk     528
PD_LBD        384
Vascular      286
Name: count, dtype: int64

In [64]:
# # How many demented cases have contributing vascular but NOT primary?
# ((first_dem_ever["CVDIF"] == 2) & 
#  (first_dem_ever["subtype_primary"] == "Other/Unk")).sum()


In [65]:
# (first_dem_ever["CVDIF"] == 2).sum()

In [66]:
# ((first_dem_ever["NACCLBDP"] == 2) &
#  (first_dem_ever["subtype_primary"] == "Other/Unk")).sum()

In [67]:
# (first_dem_ever["NACCLBDP"] == 2).sum()

# Section 1B: Check how many samples exists for different horizon years

In [68]:

import numpy as np
import pandas as pd

# Horizon range
MIN_H = 3
MAX_H = 12

results = []

print("\n=== PRE-SWEEP sanity ===")
print("baseline2 N:", baseline2["NACCID"].nunique())
print("first_dem_ever N:", first_dem_ever["NACCID"].nunique())

# Helper: build cohort-style labels for a given horizon (Approach A, matches old notebook logic)
def build_labels_A(H: int) -> pd.Series:
    # dementia within H (event-based; no requirement to have >=H follow-up)
    dem_H = first_dem_ever.loc[first_dem_ever["time_to_dem_years"] <= H, ["NACCID", "subtype_primary_contrib_any"]].copy()

    # Map subtype to ID
    subtype_map = dem_H.set_index("NACCID")["subtype_primary_contrib_any"].to_dict()
    dem_ids = set(subtype_map.keys())

    # Follow-up eligibility for *non-dementia* labels
    has_follow = baseline2.set_index("NACCID")["max_follow_years"].ge(H)

    # Label each baseline participant
    labels = []
    for nid, ok in has_follow.items():
        if nid in dem_ids:
            st = subtype_map[nid]
            labels.append(st if st in {"AD", "PD_LBD", "Vascular"} else "MixedOrOther")
        else:
            #labels.append(f"NoDementia_{H}y" if ok else "Censored")
            labels.append(f"NoDementia_Hy" if ok else "Censored")
    return pd.Series(labels, index=has_follow.index, name="label_A")

# Helper: strict (Approach B) only keeps those with >=H follow-up (so no censored)
def build_labels_B(H: int) -> pd.Series:
    eligible = baseline2.loc[baseline2["max_follow_years"].ge(H), "NACCID"]
    eligible_set = set(eligible)

    dem_H = first_dem_ever.loc[
        (first_dem_ever["time_to_dem_years"] <= H) &
        (first_dem_ever["NACCID"].isin(eligible_set)),
        ["NACCID", "subtype_primary_contrib_any"]
    ].copy()

    subtype_map = dem_H.set_index("NACCID")["subtype_primary_contrib_any"].to_dict()
    dem_ids = set(subtype_map.keys())

    labels = []
    for nid in eligible:
        if nid in dem_ids:
            st = subtype_map[nid]
            labels.append(st if st in {"AD", "PD_LBD", "Vascular"} else "MixedOrOther")
        else:
            #labels.append(f"NoDementia_{H}y")
            labels.append(f"NoDementia_Hy")
    return pd.Series(labels, index=eligible, name="label_B")

for H in range(MIN_H, MAX_H + 1):

    # =========================
    # Approach A: Event-only dementia counts (for reference)
    # =========================
    dem_within_H_A = first_dem_ever[first_dem_ever["time_to_dem_years"] <= H]
    counts_A = dem_within_H_A["subtype_primary_contrib_any"].value_counts()

    # =========================
    # Approach B: Strict eligibility dementia counts (for reference)
    # =========================
    eligible_ids = baseline2.loc[baseline2["max_follow_years"] >= H, "NACCID"]
    dem_within_H_B = first_dem_ever[
        (first_dem_ever["time_to_dem_years"] <= H) &
        (first_dem_ever["NACCID"].isin(set(eligible_ids)))
    ]
    counts_B = dem_within_H_B["subtype_primary_contrib_any"].value_counts()

    # =========================
    # Cohort-style label counts (A and B)
    # =========================
    label_A = build_labels_A(H)
    label_B = build_labels_B(H)

    vcA = label_A.value_counts()
    vcB = label_B.value_counts()

    results.append({
        "Horizon": H,

        # Dementia event counts (reference)
        "A_Total_DemEvents": len(dem_within_H_A),
        "A_AD": counts_A.get("AD", 0),
        "A_PD_LBD": counts_A.get("PD_LBD", 0),
        "A_Vascular": counts_A.get("Vascular", 0),
        "A_OtherOrUnk": counts_A.get("Other/Unk", 0),

        "B_Total_DemEvents": len(dem_within_H_B),
        "B_AD": counts_B.get("AD", 0),
        "B_PD_LBD": counts_B.get("PD_LBD", 0),
        "B_Vascular": counts_B.get("Vascular", 0),
        "B_OtherOrUnk": counts_B.get("Other/Unk", 0),

        # Eligible cohort size in B
        "B_Eligible_Cohort_Size": len(eligible_ids),

        # Cohort label counts (what training notebooks typically consume)
        f"A_NoDementia_{H}y": vcA.get(f"NoDementia_{H}y", 0),
        f"A_Censored_{H}y": vcA.get("Censored", 0),
        "A_MixedOrOther": vcA.get("MixedOrOther", 0),

        f"B_NoDementia_{H}y": vcB.get(f"NoDementia_{H}y", 0),
        "B_MixedOrOther": vcB.get("MixedOrOther", 0),
    })

horizon_df = pd.DataFrame(results)
display(horizon_df)



=== PRE-SWEEP sanity ===
baseline2 N: 38341
first_dem_ever N: 5502


,Horizon,A_Total_DemEvents,A_AD,A_PD_LBD,A_Vascular,A_OtherOrUnk,B_Total_DemEvents,B_AD,B_PD_LBD,B_Vascular,B_OtherOrUnk,B_Eligible_Cohort_Size,A_NoDementia_3y,A_Censored_3y,A_MixedOrOther,B_NoDementia_3y,B_MixedOrOther,A_NoDementia_4y,A_Censored_4y,B_NoDementia_4y,A_NoDementia_5y,A_Censored_5y,B_NoDementia_5y,A_NoDementia_6y,A_Censored_6y,B_NoDementia_6y,A_NoDementia_7y,A_Censored_7y,B_NoDementia_7y,A_NoDementia_8y,A_Censored_8y,B_NoDementia_8y,A_NoDementia_9y,A_Censored_9y,B_NoDementia_9y,A_NoDementia_10y,A_Censored_10y,B_NoDementia_10y,A_NoDementia_11y,A_Censored_11y,B_NoDementia_11y,A_NoDementia_12y,A_Censored_12y,B_NoDementia_12y
0,3,2929,2272,225,105,327,1598,1274,120,48,156,17724,0.0,19286.0,327,0.0,156,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,3526,2729,269,144,384,1559,1256,102,61,140,14853,NaN,NaN,384,NaN,140,0.0,21521.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,3958,3068,297,172,421,1391,1140,71,57,123,12474,NaN,NaN,421,NaN,123,NaN,NaN,NaN,0.0,23300.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,4286,3320,311,200,455,1146,931,47,57,111,10147,NaN,NaN,455,NaN,111,NaN,NaN,NaN,NaN,NaN,NaN,0.0,25054.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,4550,3538,327,214,471,955,778,42,47,88,8139,NaN,NaN,471,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,26607.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,8,4737,3681,342,233,481,782,628,34,43,77,6595,NaN,NaN,481,NaN,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,27791.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9,4891,3811,352,241,487,631,509,29,31,62,5404,NaN,NaN,487,NaN,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,28677.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,10,5020,3909,361,255,495,484,380,25,28,51,4412,NaN,NaN,495,NaN,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,29393.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
8,11,5118,3989,367,260,502,380,300,20,22,38,3582,NaN,NaN,502,NaN,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,30021.0,0.0,NaN,NaN,NaN
9,12,5207,4060,369,270,508,288,231,11,16,30,2831,NaN,NaN,508,NaN,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,30591.0,0.0


# Section 2A
# Section 2B Tried adding various features to see how it impacts the scores

In [69]:
import os
import numpy as np
import pandas as pd

# === Build cohort.csv for a chosen horizon (matches old notebook labeling + training-ready export) ===

HORIZON_YEARS = 5  # change as needed
H = HORIZON_YEARS

# ----------------------------
# 1) Dementia within H (event window)
# ----------------------------
dem_H = first_dem_ever.loc[
    first_dem_ever["time_to_dem_years"] <= H,
    ["NACCID", "dementia_date", "time_to_dem_years",
     "subtype_primary_contrib_any", "subtype_label_source_3tier"]
].copy()

# ----------------------------
# 2) Merge onto baseline2 and assign labels (old notebook style)
# ----------------------------
cohort = baseline2.merge(dem_H, on="NACCID", how="left")

cohort["has_dementia_H"] = cohort["dementia_date"].notna()
cohort["has_H_followup"] = cohort["max_follow_years"] >= H

def assign_label(r):
    if r["has_dementia_H"]:
        st = r.get("subtype_primary_contrib_any", "Other/Unk")
        if st in {"AD", "PD_LBD", "Vascular"}:
            return st
        return "MixedOrOther"
    if r["has_H_followup"]:
        #return f"NoDementia_{H}y"
        return f"NoDementia_Hy"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)

print("Cohort rows:", len(cohort))
display(cohort["label"].value_counts(dropna=False))

# ----------------------------
# 3) Build training export X (like old notebook)
# ----------------------------
STARTER_FEATURES = [
    "NACCID","baseline_date",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD",

    # Vascular risk / history (baseline-safe, non-leaky)
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",

    # Medication proxies for vascular risk treatment (baseline-safe)
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",

    # Other helpful baseline covariates
    "APNEA","DEP2YRS",

    # Cognition / function
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",

        # PD/LBD symptom block (UDS B9 clinician judgment)
    "COGFLUC",   # fluctuating cognition (0/1/9)
    "BEVHALL",   # visual hallucinations (0/1/9)
    # "BEAHALL",   # auditory hallucinations (0/1/9)
    # "BEDEL",     # delusions (0/1/9)
    # "BEAGIT",    # agitation (0/1/9)
    "BEREM",     # REM sleep behavior disorder (0/1/9)
    # "BEDEP",     # depressed mood (0/1/9)
    # "BEANX",     # anxiety (0/1/9)
    # "BEAPATHY",  # apathy/withdrawal (0/1/9)
    #"STROKMUL",
    #"PREVSTK",
    #"STROKDEC","STKIMAG",
    # "STROKE",
    #"STROKIF",
    # Motor / Parkinsonism exam block (UDS B8)
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
]

# baseline should contain baseline_date + baseline features
cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

# Merge label and follow-up info onto X
X = X.merge(
    cohort[["NACCID","baseline_date","label","max_follow_years","has_H_followup","has_dementia_H"]],
    on=["NACCID","baseline_date"],
    how="left"
)

# for v in ["COGFLUC","BEVHALL","BEREM"]:
#     if v in X.columns:
#         print("\n", v)
#         print(pd.crosstab(X["label"], X[v], normalize="index"))
#     else:
#         print(f"{v} not in X.columns")

# VASC_VARS = ["CBSTROKE","STROKMUL","PREVSTK","STROKDEC","STKIMAG","STROKE","STROKIF"]

# present = [c for c in VASC_VARS if c in X.columns]
# missing = [c for c in VASC_VARS if c not in X.columns]

# print("Vascular vars present:", present)
# print("Vascular vars missing:", missing)

# # Focus the crosstab on the main modeling labels (optional but recommended)
# MAIN_LABELS = ["NoDementia_Hy","AD","PD_LBD","Vascular"]
# Xk = X[X["label"].isin(MAIN_LABELS)].copy()

# for v in present:
#     print("\n====================")
#     print("VAR:", v)
#     # show raw distribution by label (includes -4/8/9)
#     try:
#         print(pd.crosstab(Xk["label"], Xk[v], normalize="index"))
#     except Exception as e:
#         print("crosstab failed:", e)
#     # show missingness rate
#     print("Missing (NaN) rate:", Xk[v].isna().mean() if v in Xk.columns else None)

# # Make binary helper columns just for analysis (do not permanently keep if you don't want yet)
# Xb = Xk.copy()

# # CBSTROKE: 1 recent or 2 remote means stroke history
# if "CBSTROKE" in Xb.columns:
#     Xb["CBSTROKE_any"] = Xb["CBSTROKE"].isin([1,2]).astype(float)
#     # keep NA for -4/9 if you want
#     Xb.loc[Xb["CBSTROKE"].isin([-4,9]), "CBSTROKE_any"] = np.nan

# # PREVSTK: previous symptomatic stroke yes/no
# if "PREVSTK" in Xb.columns:
#     Xb["PREVSTK_yes"] = (Xb["PREVSTK"] == 1).astype(float)
#     Xb.loc[Xb["PREVSTK"].isin([-4]), "PREVSTK_yes"] = np.nan

# # STROKDEC: stroke temporally related (only defined if prev stroke; 8 means no prev stroke)
# if "STROKDEC" in Xb.columns:
#     Xb["STROKDEC_yes"] = (Xb["STROKDEC"] == 1).astype(float)
#     Xb.loc[Xb["STROKDEC"].isin([-4,8]), "STROKDEC_yes"] = np.nan

# # STKIMAG: imaging confirmed stroke (8 means no prev stroke; 9 means unknown imaging)
# if "STKIMAG" in Xb.columns:
#     Xb["STKIMAG_yes"] = (Xb["STKIMAG"] == 1).astype(float)
#     Xb.loc[Xb["STKIMAG"].isin([-4,8,9]), "STKIMAG_yes"] = np.nan

# # Now see prevalence by label (mean = prevalence)
# for v in ["CBSTROKE_any","PREVSTK_yes","STROKDEC_yes","STKIMAG_yes"]:
#     if v in Xb.columns:
#         print("\n", v)
#         print(Xb.groupby("label")[v].mean().sort_values(ascending=False))

# Backward-compat convenience column if your training expects has_5y_followup
# (only create when H=5)
# if H == 5:
#     X["has_5y_followup"] = X["has_H_followup"]


# ----------------------------
# 4A) UDS missing-code cleansing (MUST run unconditionally)
# ----------------------------

# Common UDS special missing / not-applicable codes
# -4 is the big one; 8/9 often used for NA/Unknown for coded fields.
GLOBAL_MISSING = {-4}

# For many binary/coded clinical vars: 8/9 represent NA/Unknown (dictionary-dependent).
CODED_MISSING = {-4, 8, 9}

# For some test score vars, 88/99 can appear as NA/Unknown depending on instrument.
# We'll apply these ONLY to the clearly numeric cognitive tests listed below.
NUMERIC_TEST_EXTRA_MISSING = {88, 99, 888, 999}

# 1) Force numeric vars to numeric and replace special codes with NaN
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"]

for v in NUM_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")

        # Always treat -4 as missing
        X.loc[X[v].isin(GLOBAL_MISSING), v] = np.nan

        # For cognitive test-like vars, also treat 88/99/... as missing
        if v in {"NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"}:
            X.loc[X[v].isin(NUMERIC_TEST_EXTRA_MISSING), v] = np.nan

# 2) For coded/binary medical history + meds vars, replace -4/8/9 with NaN
CODED_VARS = [
    "SEX","RACE","HISPANIC","NACCUDSD",
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
    "APNEA","DEP2YRS",
    "COGFLUC","BEVHALL", "BEREM",
    # "STROKE",
    #"PREVSTK",
    #"BEAHALL","BEDEL","BEAGIT","BEDEP","BEANX","BEAPATHY",
    # Motor block
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
]

for v in CODED_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")
        X.loc[X[v].isin(CODED_MISSING), v] = np.nan

# 3) (Optional but useful) add missing indicators for the coded vars too
# This helps tree models capture "missingness" without using -4 as a real value.
for v in CODED_VARS:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)



# ----------------------------
# 4) Clean coded numerics + add missing indicators (if helpers exist)
# ----------------------------
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB"]

for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

# ----------------------------
# 4B) Vascular engineered features (baseline-safe)
# ----------------------------

# Helper: safely coerce a column to numeric if it exists
# def _numcol(df, c):
#     if c in df.columns:
#         df[c] = pd.to_numeric(df[c], errors="coerce")

# # 1) TRAIL ratio (executive dysfunction proxy)
# # Vascular often shows disproportionate executive dysfunction.
# if "TRAILA" in X.columns and "TRAILB" in X.columns:
#     _numcol(X, "TRAILA")
#     _numcol(X, "TRAILB")
#     # Avoid divide-by-zero
#     X["trail_ratio"] = X["TRAILB"] / X["TRAILA"].replace({0: np.nan})
# else:
#     # optional: keep column absent; nothing to do
#     pass

# # 2) Executive-minus-memory contrast
# # Uses TRIALB as exec proxy; MMSE/MOCA as global cognition proxy
# # (Higher value = worse executive relative to global cognition)
# if "TRAILB" in X.columns:
#     _numcol(X, "TRAILB")

# # Choose a global cognition score: prefer MOCA if present, else MMSE
# global_cog = None
# if "NACCMOCA" in X.columns:
#     _numcol(X, "NACCMOCA")
#     global_cog = "NACCMOCA"
# elif "NACCMMSE" in X.columns:
#     _numcol(X, "NACCMMSE")
#     global_cog = "NACCMMSE"

# if global_cog is not None and "TRAILB" in X.columns:
#     # z-score each (within cohort) so scales are comparable
#     # For TRAILB, higher is worse; for MOCA/MMSE, lower is worse → invert cognition
#     z_trailb = (X["TRAILB"] - X["TRAILB"].mean()) / (X["TRAILB"].std(ddof=0) + 1e-9)
#     z_cog_inv = ((X[global_cog].mean() - X[global_cog]) / (X[global_cog].std(ddof=0) + 1e-9))

#     X["exec_minus_memory"] = z_trailb - z_cog_inv

# # 3) Vascular burden score (simple count of risk factors present)
# # This captures cumulative cerebrovascular risk better than individual flags.
# RISK_FLAGS = [
#     "DIABETES", "HYPERTEN", "HYPERCHO", "CVAFIB", "CBSTROKE", "CBTIA", "CVCHF"
# ]
# present_flags = [c for c in RISK_FLAGS if c in X.columns]

# for c in present_flags:
#     _numcol(X, c)
#     # after cleansing, "1" should mean present; keep NaN as NaN

# if present_flags:
#     # Convert each flag to 1 if ==1 else 0, but preserve NaN if the entire row is NaN across all flags
#     flag_mat = pd.DataFrame(index=X.index)
#     for c in present_flags:
#         flag_mat[c] = np.where(X[c].isna(), np.nan, (X[c] == 1).astype(float))

#     # If all flags are missing, burden should be NaN; else sum ignoring NaNs
#     all_missing = flag_mat.isna().all(axis=1)
#     X["vascular_burden"] = flag_mat.sum(axis=1, skipna=True)
#     X.loc[all_missing, "vascular_burden"] = np.nan

# # 4) Optional: add an interaction capturing "exec bad but global cognition preserved"
# # Often a vascular pattern vs AD.
# if global_cog is not None and "TRAILB" in X.columns:
#     # Higher = more vascular-like (bad trails + still decent global cog)
#     X["exec_preserved_cog"] = z_trailb + (X[global_cog] - X[global_cog].mean()) / (X[global_cog].std(ddof=0) + 1e-9)

# print("Added engineered vascular features:",
#       [c for c in ["trail_ratio","exec_minus_memory","vascular_burden","exec_preserved_cog"] if c in X.columns])

# display(X.head())

# ----------------------------
# 5) Sanity: vascular-signal vars present?
# ----------------------------
# vas_candidates = ["CBSTROKE","CBTIA","HYPERTEN","DIABETES","HYPERCHO","CVAFIB",
#                   "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
#                   "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]

# print("Present vascular-signal vars in cohort export:",
#       [v for v in vas_candidates if v in X.columns])
# print("Missing vascular-signal vars in this export:",
#       [v for v in vas_candidates if v not in X.columns])

# Create clean vascular binary features

# if "CBSTROKE" in X.columns:
#     X["CBSTROKE_any"] = X["CBSTROKE"].isin([1,2]).astype(float)
#     X.loc[X["CBSTROKE"].isin([-4,9]), "CBSTROKE_any"] = np.nan

# X = X.drop(columns=["CBSTROKE"], errors="ignore")

# if "PREVSTK" in X.columns:
#     X["PREVSTK_yes"] = (X["PREVSTK"] == 1).astype(float)
#     X.loc[X["PREVSTK"].isin([-4]), "PREVSTK_yes"] = np.nan

# if "STROKE" in X.columns:
#     X["STROKE_yes"] = (X["STROKE"] == 1).astype(float)
#     X.loc[X["STROKE"].isin([-4]), "STROKE_yes"] = np.nan

# ----------------------------
# 6) Optional: merge nearest CSF biomarkers to baseline (if path exists)
# ----------------------------

NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")

if "NACC_CSF_PATH" in globals() and os.path.exists(NACC_CSF_PATH):
    csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
    csf["NACCID"] = csf["NACCID"].astype(str)

    for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
        if c in csf.columns:
            csf[f"{c}_missing"] = csf[c].isna().astype(int)

    def make_csf_date(df_: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
        m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
        d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
        return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

    csf["csf_date"] = make_csf_date(csf)

    csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
    csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
    csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

    csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

    merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
    merge_cols = [c for c in merge_cols if c in csf_nearest.columns]
    X = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")

    print("Merged CSF. Rows:", len(X))
    if "CSFABETA" in X.columns:
        print("CSFABETA non-null:", X["CSFABETA"].notna().sum())
else:
    print("CSF file not found (or NACC_CSF_PATH not set); skipping CSF merge.")

#Remove unnecessary _missing flags added earlier
IMPORTANT_MISSING = [
    "TRAILA",
    "TRAILB",
    "NACCMMSE",
    "NACCMOCA",
    "CDRSUM"
]

for v in IMPORTANT_MISSING:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

drop_missing_cols = [
    c for c in X.columns
    if c.endswith("_missing") and
       c.replace("_missing","") not in IMPORTANT_MISSING
]

X = X.drop(columns=drop_missing_cols)



Cohort rows: 38341


label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64

Merged CSF. Rows: 38341
CSFABETA non-null: 1559


In [70]:
# pdlbd_cols = ["COGFLUC","BEVHALL","BEAHALL","BEDEL","BEAGIT","BEREM","BEDEP","BEANX","BEAPATHY"]
# present = [c for c in pdlbd_cols if c in X.columns]

# print("PD/LBD symptom vars present:", present)
# if present:
#     display(X[present].describe(include="all").T)
#     print("Missingness rates:")
#     display(X[present].isna().mean().sort_values(ascending=False))

In [71]:
# ----------------------------
# 7) Save outputs
# ----------------------------



out_full = f"cohort_full_H{H}.csv"   # full baseline2 + label + event info (debug)
out_XH   = f"cohort_H{H}.csv"        # training export for this horizon
out_X    = "cohort.csv"              # default name for training notebooks

out_full_path = os.path.join(OUT_DIR, out_full)
out_XH_path = os.path.join(OUT_DIR, out_XH)
out_X_path = os.path.join(OUT_DIR, out_X)

cohort.to_csv(out_full_path, index=False)
X.to_csv(out_XH_path, index=False)
X.to_csv(out_X_path, index=False)

print("Wrote:", out_full, out_XH, "and", out_X)
print("Training export label counts:")
display(X["label"].value_counts(dropna=False))

Wrote: cohort_full_H5.csv cohort_H5.csv and cohort.csv
Training export label counts:


label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64

In [72]:
print("Number of modeling columns:", len(X.columns))
print(sorted(X.columns))

Number of modeling columns: 63
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'CSFABETA', 'CSFPTAU', 'CSFTTAU', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'GAITDIS', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'NACCAC', 'NACCAGE', 'NACCANGI', 'NACCBETA', 'NACCCCBS', 'NACCDIUR', 'NACCID', 'NACCLIPL', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCUDSD', 'NACCVASD', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SLOWINGL', 'SLOWINGR', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'abs_days_from_baseline', 'baseline_date', 'has_H_followup', 'has_dementia_H', 'label', 'max_follow_years']
